In [1]:
%run ./methods.ipynb

loaded 920 rows, kept 740 after filtering
all the data checks look good


all 12 method checks look good


In [2]:
# picking the learning rate. 0.05 hadn't finished training in 250 steps, so we take the smallest lr
# that gets within 0.02 of a long run's training loss (training data only, averaged over 5 seeds)
def train_loss(models):
    tot, n = 0.0, 0
    for s in sites:
        p = np.clip(predict_proba(models[s.cid], s.X_train), 1e-9, 1 - 1e-9)
        tot += -(s.y_train * np.log(p) + (1 - s.y_train) * np.log(1 - p)).sum(); n += len(p)
    return tot / n

ref = np.mean([train_loss(fit(sites, "fedavg", TrainConfig(lr=0.05, rounds=2000), rep=r, run_key=f"lrsel-ref|{r}").models) for r in range(5)])
rows = []
for lr in [0.05, 0.1, 0.2, 0.3, 0.5]:
    l = np.mean([train_loss(fit(sites, "fedavg", TrainConfig(lr=lr), rep=r, run_key=f"lrsel|{lr}|{r}").models) for r in range(5)])
    rows.append({"lr": lr, "train_loss": l, "gap": l - ref, "ok": l - ref <= 0.02})
print("reference loss:", round(ref, 4))
pd.DataFrame(rows).round(4)

reference loss: 0.4278


,lr,train_loss,gap,ok
0,0.05,0.5007,0.0729,False
1,0.10,0.4662,0.0384,False
2,0.20,0.4481,0.0203,False
3,0.30,0.4421,0.0142,True
4,0.50,0.4374,0.0096,True


In [3]:
# how much noise each hospital needs. smaller hospitals use more of their data each step, so they need a lot more
tcfg = TrainConfig(**CONFIG["train"])
rows = []
for s in sites:
    q = tcfg.batch / len(s.y_train)
    for e in CONFIG["experiment"]["epsilons"]:
        rows.append({"hospital": s.name, "eps": e, "sigma": calibrate_sigma(e, tcfg.delta, q, tcfg.steps)})
pd.DataFrame(rows).pivot(index="hospital", columns="eps", values="sigma").round(3)

eps,1,2,4,8
hospital,,,,
Cleveland,5.312,2.930,1.724,1.114
Hungary,6.133,3.350,1.934,1.221
Switzerland,34.375,18.203,9.844,5.454
VA Long Beach,12.188,6.543,3.604,2.086


In [4]:
# epsilon keeps adding up over all 50 rounds and never resets (one fedavg run aiming for eps 2)
clients = [Client(s.cid, s.name, s.X_train, s.y_train, tcfg, "dp", 2.0, "demo-acct") for s in sites]
w = {c.cid: c.N / sum(k.N for k in clients) for c in clients}
glob, rows = init_theta(13, 1000), []
for r in range(tcfg.rounds):
    for c in clients:
        c.run_local(glob, tcfg.local_steps)
    glob = sum(w[c.cid] * c.theta for c in clients)
    if r == 0 or (r + 1) % 10 == 0:
        rows.append({"round": r + 1, **{c.name: c.accountant.epsilon() for c in clients}})
pd.DataFrame(rows).round(3)

,round,Cleveland,Hungary,Switzerland,VA Long Beach
0,1,0.316,0.301,0.243,0.263
1,10,0.878,0.869,0.832,0.845
2,20,1.242,1.235,1.211,1.219
3,30,1.530,1.525,1.510,1.516
4,40,1.777,1.774,1.768,1.768
5,50,1.999,1.997,1.998,1.996


In [5]:
# every setting we run, plus the runner. it saves to results/ and skips anything already done
NONPRIVATE = "nonprivate"


def config_hash(cfg):
    return hashlib.sha256(json.dumps({k: cfg[k] for k in ("train", "data")}, sort_keys=True).encode()).hexdigest()[:12]


def setting(group, method, eps=None, mode=None, lam=0.0, weighting="size", transform="fixed", chol_policy="keep_zero", frac=1.0):
    mode = mode or (NONPRIVATE if eps is None else "dp")
    return dict(group=group, method=method, mode=mode, eps=eps, lam=lam, weighting=weighting,
                transform=transform, chol_policy=chol_policy, frac=frac)


def build_matrix(cfg):
    E, lam = cfg["experiment"], cfg["experiment"]["lambda"]
    S = [setting("core", "pooled")]
    for m in ("local", "fedavg", "mrmtl"):
        for e in [None] + E["epsilons"]:
            S.append(setting("core", m, e, lam=lam if m == "mrmtl" else 0.0))
    S.append(setting("core", "fedavg", mode="clip_only"))
    for l in E["lambda_sweep"]["values"]:
        if l != lam:
            S += [setting("lambda_sweep", "mrmtl", e, lam=l) for e in E["lambda_sweep"]["eps"]]
    for m in ("fedavg", "mrmtl"):
        S += [setting("equal_weights", m, e, lam=lam if m == "mrmtl" else 0.0, weighting="equal") for e in E["equal_weights"]["eps"]]
    for m in ("local", "fedavg", "mrmtl"):
        S += [setting("chol_sensitivity", m, e, lam=lam if m == "mrmtl" else 0.0, chol_policy="zero_as_missing")
              for e in E["chol_sensitivity"]["eps"]]
    S.append(setting("flamby_parity", "pooled", transform="flamby_pooled"))
    S += [setting("flamby_parity", m, transform="flamby_center") for m in ("local", "fedavg")]
    for f in E["size_analysis"]["fractions"]:
        if f != 1.0:
            for m in ("local", "fedavg", "mrmtl"):
                S += [setting("size_analysis", m, e, lam=lam if m == "mrmtl" else 0.0, frac=f) for e in E["size_analysis"]["eps"]]
    return S


def setting_key(s):
    return json.dumps(s, sort_keys=True)


def run_id(s, rep, chash):
    return hashlib.sha256(f"{setting_key(s)}|{rep}|{chash}".encode()).hexdigest()[:16]


def setting_name(s):
    parts = [s["method"], (NONPRIVATE if s["eps"] is None else f"eps={s['eps']:g}") if s["mode"] != "clip_only" else "clip_only"]
    if s["method"] == "mrmtl":
        parts.append(f"lam={s['lam']:g}")
    if s["weighting"] != "size":
        parts.append("equal_w")
    if s["chol_policy"] != "keep_zero":
        parts.append("chol_missing")
    if s["transform"] != "fixed":
        parts.append(s["transform"])
    if s["frac"] != 1.0:
        parts.append(f"cleveland_frac={s['frac']:g}")
    return " | ".join(parts)


_site_cache = {}

def get_sites(s, cfg):
    key = (s["transform"], s["chol_policy"], s["frac"])
    if key not in _site_cache:
        frac = {cfg["experiment"]["size_analysis"]["site"]: s["frac"]} if s["frac"] != 1.0 else None
        _site_cache[key] = build_sites(retained, transform=s["transform"], chol_policy=s["chol_policy"], train_fraction=frac)
    return _site_cache[key]


def _append(path, rows):
    if rows:
        pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)


def run_matrix(cfg, resume=True):
    out = ROOT / cfg["output_dir"]
    out.mkdir(parents=True, exist_ok=True)
    chash, tcfg, matrix = config_hash(cfg), TrainConfig(**cfg["train"]), build_matrix(cfg)
    reps = range(cfg["experiment"]["repetitions"])
    runs_path, pred_path, ledger_path = out / "runs.csv", out / "predictions.csv", out / "privacy_ledger.csv"
    done = set()
    if resume and runs_path.exists():
        old = pd.read_csv(runs_path)
        done = set(old.loc[(old.config_hash == chash) & (old.status == "complete"), "run_id"])
    elif not resume:
        for p in (runs_path, pred_path, ledger_path):
            p.unlink(missing_ok=True)
    pd.DataFrame([{"run_id": run_id(s, r, chash), "setting": setting_name(s), "group": s["group"], "repetition": r}
                  for s in matrix for r in reps]).to_csv(out / "manifest.csv", index=False)

    n_new = 0
    for s in matrix:
        for r in reps:
            rid = run_id(s, r, chash)
            if rid in done:
                continue
            st = get_sites(s, cfg)
            base = {"run_id": rid, "config_hash": chash, "dataset_version": UCI_ZIP_SHA256[:12],
                    "preprocessing_version": "fixed-bounds-v1" if s["transform"] == "fixed" else s["transform"],
                    "split_id": f"flamby-{FLAMBY_COMMIT[:7]}-rs43", "setting": setting_name(s), "group": s["group"],
                    "training_repetition": r, "initialization_seed": 1000 + r, "method": s["method"], "mode": s["mode"],
                    "epsilon_target": s["eps"], "delta": tcfg.delta if s["mode"] == "dp" else None, "dp_enabled": s["mode"] == "dp",
                    "lambda": s["lam"] if s["method"] == "mrmtl" else None, "weighting": s["weighting"],
                    "chol_policy": s["chol_policy"], "train_fraction_site": s["frac"],
                    "rounds": tcfg.rounds if s["method"] != "pooled" else None,
                    "local_steps": tcfg.local_steps if s["method"] != "pooled" else None}
            try:
                res = fit(st, s["method"], tcfg, mode=s["mode"], eps=s["eps"], lam=s["lam"], weighting=s["weighting"],
                          rep=r, run_key=f"{setting_key(s)}|{r}|{chash}")
                rows, preds = evaluate(res.models, st)
                ledger = {l["hospital_id"]: l for l in res.ledger}
                out_rows = []
                for row in rows:
                    l = ledger.get(row["hospital_id"], ledger.get(-1))
                    out_rows.append({**base, **row, "epsilon_realized": l["epsilon_realized"], "clip_norm": l["clip_norm"],
                                     "noise_multiplier": l["noise_multiplier"], "sample_rate": l["sample_rate"],
                                     "optimizer_steps": l["optimizer_steps"], "patient_grad_evals": l["patient_grad_evals"],
                                     "aggregation_weight": res.weights.get(row["hospital_id"]), "elapsed_seconds": res.elapsed,
                                     "status": "complete", "failure_reason": ""})
                pred_rows = [{"run_id": rid, "setting": setting_name(s), "group": s["group"], "repetition": r, **p} for p in preds]
                led_rows = [{"run_id": rid, "setting": setting_name(s), "repetition": r, "epsilon_target": s["eps"], **l} for l in res.ledger]
                if s["mode"] == "dp":
                    assert all(l["epsilon_realized"] <= s["eps"] + 1e-9 and l["accountant_steps"] == tcfg.steps for l in res.ledger)
            except Exception as ex:  # we keep failed runs in the table instead of hiding them
                out_rows, pred_rows, led_rows = [{**base, "status": "failed", "failure_reason": repr(ex)[:300]}], [], []
                traceback.print_exc()
            _append(runs_path, out_rows); _append(pred_path, pred_rows); _append(ledger_path, led_rows)
            n_new += 1
    (out / "environment.json").write_text(json.dumps({
        "python": platform.python_version(), "torch": torch.__version__, "opacus": opacus.__version__, "numpy": np.__version__,
        "pandas": pd.__version__, "sklearn": sklearn.__version__, "platform": platform.platform(), "config_hash": chash,
        "uci_zip_sha256": UCI_ZIP_SHA256, "flamby_reference_commit": FLAMBY_COMMIT}, indent=2))
    print(f"ran {n_new} new runs, skipped {len(done)} already done")


def check_complete(cfg):
    out = ROOT / cfg["output_dir"]
    man, runs = pd.read_csv(out / "manifest.csv"), pd.read_csv(out / "runs.csv")
    assert runs[runs.status == "complete"].duplicated(["run_id", "hospital_id"]).sum() == 0
    return man[~man.run_id.isin(set(runs.loc[runs.status == "complete", "run_id"]))]

In [6]:
# run everything. takes about 3 min the first time
run_matrix(CONFIG, resume=True)
missing = check_complete(CONFIG)
print(len(pd.read_csv(RESULTS / "manifest.csv")), "runs expected,", len(missing), "missing or failed")

ran 0 new runs, skipped 255 already done
255 runs expected, 0 missing or failed
